## v5 — all data, three seeds, ensembled

The best result. Two changes from v4: train on all 1693 samples with no
held-out split, and average three models trained from different seeds. Dropping
to a squashed 384x384 input made each run 3x faster, which is what bought the
budget for three of them. The last two cells are the end-of-hackathon
submission blending.

---

**Provenance.** This is an as-run artifact from the BrabantHack 2026 DEMCON
challenge, preserved with its original outputs so the numbers quoted in
[`docs/experiments.md`](../docs/experiments.md) can be checked against the logs
that produced them. It ran on a shared GPU server, so the paths (`/home/y2a/...`) and the `CUDA_VISIBLE_DEVICES` pin are the machine's, not yours.

The method it arrived at has since been rewritten as a tested package under
[`src/shadow_detection/`](../src/shadow_detection); run
`shadow-detection train --help` rather than this notebook.

Inline comments in the final two cells were translated from Russian; nothing else
was altered. The preserved traceback in the second-to-last cell still quotes the
original line, because editing a captured error output would misrepresent the run.

Note that that cell **failed**: the other sub-team's CSV was not in the working
directory, so the 0.7/0.3 cross-team blend it describes produced no file here.
The final cell -- a weighted blend across our own five submissions -- did run.


# V5: Full Data Training + 3-Seed Ensemble

**Two tricks for max score:**
1. Train on ALL 1692 samples (no val holdout) — 15% more data
2. Train 3 models with different seeds → average predictions

Ensemble of 3 models almost always beats single model by 2-5%.
Training 3x takes ~30 min total with their fast architecture.

In [16]:
import json, os, random, time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
from scipy.ndimage import sobel, gaussian_filter
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

os.environ['CUDA_VISIBLE_DEVICES'] = '4'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

TRAIN_DIR = Path('/home/y2a/detection-by-shadow/train_data/train_data')
TEST_DIR = Path('/home/y2a/detection-by-shadow/test_data/test_data')
SAMPLE_CSV = Path('/home/y2a/detection-by-shadow/submission_example.csv')
SAVE_DIR = Path('/home/y2a/Hackaton/shadow_detection')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_W, IMG_H = 720, 480
INPUT_SIZE = (384, 384)
BATCH_SIZE = 128
NUM_EPOCHS = 40           # no early stopping, fixed epochs
LR = 3e-3
BACKBONE_LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
SEEDS = [42, 123, 777]    # 3 different seeds for ensemble

Device: cuda
GPU: NVIDIA RTX 6000 Ada Generation


In [17]:
def extract_geometric_features(img_array):
    if len(img_array.shape)==3 and img_array.shape[2]==4: img_array=img_array[:,:,:3]
    gray = np.mean(img_array.astype(np.float32), axis=2)
    H, W = gray.shape
    rm = np.median(gray, axis=1, keepdims=True)
    sm = gray < (rm * 0.85)
    rs = int(H*0.4); road = sm[rs:,:]
    f = np.zeros(19, dtype=np.float32)
    f[0]=np.mean(gray[int(H*0.625):,:20])/255; f[1]=np.mean(gray[int(H*0.625):,-20:])/255
    f[2]=f[0]/(f[1]+1e-6)
    ys,xs=np.where(road)
    if len(xs)>50:
        f[3]=np.mean(xs)/W; f[4]=(np.mean(ys)+rs)/H; f[5]=np.std(xs)/W; f[6]=np.std(ys)/H
        f[7]=len(xs)/(road.shape[0]*road.shape[1])
        lm=np.sum(road[:,:W//2]); rm2=np.sum(road[:,W//2:]); f[8]=lm/(lm+rm2+1e-6)
        cd=np.sum(road,axis=0).astype(float); f[9]=np.argmax(cd)/W
        f[10]=np.average(np.arange(W).astype(float),weights=cd+1e-6)/W
        f[11]=np.sum(road[:,:30])/(road.shape[0]*30); f[12]=np.sum(road[:,-30:])/(road.shape[0]*30)
        if len(xs)>100:
            cov=np.cov(xs-np.mean(xs),ys-np.mean(ys)); _,ev=np.linalg.eigh(cov)
            f[13]=np.arctan2(ev[1,1],ev[0,1])/np.pi
    else: f[3:14]=0.5
    bl2=gaussian_filter(gray,sigma=20); sd=bl2-gray; r2=sd[int(H*0.58):,:]; m2=r2>8
    y2,x2=np.where(m2)
    if len(x2)>50:
        f[14]=np.mean(r2[m2])/100; f[15]=np.max(r2[m2])/100
        f[16]=np.mean(np.abs(sobel(r2,axis=1))[m2])/50
        f[17]=len(x2)/((np.max(y2)-np.min(y2)+1)*(np.max(x2)-np.min(x2)+1)+1e-6)
        f[18]=np.percentile(r2[m2],90)/100
    return f

print('Features ready')

Features ready


In [18]:
# Load ALL annotations + precompute geo
def load_all(data_dir):
    png_files = {p.stem: p for p in data_dir.rglob('*.png')}
    samples = []
    t0 = time.time()
    json_files = sorted(data_dir.rglob('*.json'))
    for i, jf in enumerate(json_files):
        if i%300==0: print(f'  {i}/{len(json_files)} ({time.time()-t0:.0f}s)')
        with open(jf) as f: ann = json.load(f)
        fname = ann['file_name']
        if fname not in png_files: continue
        tl, br = ann['bbox']['top_left'], ann['bbox']['bottom_right']
        xmin, ymin, xmax, ymax = tl[0], tl[1], br[0], br[1]
        w, h = xmax-xmin, ymax-ymin
        yc = (ymin+ymax)/2
        side = 0 if xmin < 0 else 1
        dist = abs(xmin) if side == 0 else xmax - IMG_W
        img = np.array(Image.open(png_files[fname]).convert('RGB'))
        geo = extract_geometric_features(img)
        samples.append({
            'img_path': str(png_files[fname]), 'file_name': fname,
            'side': side, 'distance_from_edge': dist,
            'bbox_width': w, 'bbox_height': h, 'y_center': yc,
            'direction': ann['walking_into_frame_bool'], 'geo': geo,
        })
    return samples

print('Loading ALL data...')
all_samples = load_all(TRAIN_DIR)
print(f'Total: {len(all_samples)} samples')

# Normalization stats from ALL data
target_keys = ['distance_from_edge', 'bbox_width', 'bbox_height', 'y_center']
target_stats = {}
for k in target_keys:
    vals = np.array([s[k] for s in all_samples])
    target_stats[k] = {'mean': float(vals.mean()), 'std': float(vals.std())}
    print(f'  {k:>20s}: mean={vals.mean():.2f}, std={vals.std():.2f}')

def norm_t(v, k): return (v - target_stats[k]['mean']) / (target_stats[k]['std'] + 1e-8)
def denorm_t(v, k): return v * (target_stats[k]['std'] + 1e-8) + target_stats[k]['mean']

Loading ALL data...
  0/1693 (0s)
  300/1693 (19s)
  600/1693 (38s)
  900/1693 (57s)
  1200/1693 (76s)
  1500/1693 (94s)
Total: 1693 samples
    distance_from_edge: mean=208.59, std=45.14
            bbox_width: mean=80.87, std=29.86
           bbox_height: mean=172.91, std=34.67
              y_center: mean=309.34, std=14.27


In [19]:
class ShadowDatasetV5(Dataset):
    def __init__(self, samples, target_stats, augment=False):
        self.samples = samples
        self.ts = target_stats
        self.augment = augment
        self.base_tf = transforms.Compose([
            transforms.Resize(INPUT_SIZE), transforms.ToTensor(),
            transforms.Normalize([0.422,0.413,0.394],[0.167,0.174,0.233])])
        self.aug_tf = transforms.Compose([
            transforms.Resize(INPUT_SIZE),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.GaussianBlur(3, sigma=(0.1,1.0)),
            transforms.ToTensor(),
            transforms.Normalize([0.422,0.413,0.394],[0.167,0.174,0.233])])
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['img_path']).convert('RGB')
        side = s['side']
        direction = s['direction']
        geo = s['geo'].copy()
        
        do_flip = self.augment and random.random() < 0.5
        if do_flip:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            side = 1 - side
            direction = 1 - direction
            geo[0],geo[1] = geo[1],geo[0]; geo[2]=geo[0]/(geo[1]+1e-6)
            geo[3]=1-geo[3]; geo[8]=1-geo[8]; geo[9]=1-geo[9]; geo[10]=1-geo[10]
            geo[11],geo[12] = geo[12],geo[11]
        
        img = self.aug_tf(img) if self.augment else self.base_tf(img)
        
        reg = torch.tensor([
            norm_t(s['distance_from_edge'], 'distance_from_edge'),
            norm_t(s['bbox_width'], 'bbox_width'),
            norm_t(s['bbox_height'], 'bbox_height'),
            norm_t(s['y_center'], 'y_center'),
        ], dtype=torch.float32)
        
        return (img, torch.tensor(side, dtype=torch.long), reg,
                torch.tensor(direction, dtype=torch.long),
                torch.from_numpy(geo).float())

print('Dataset ready')

Dataset ready


In [20]:
class ShadowModelV5(nn.Module):
    def __init__(self, num_geo=19, dropout=0.3):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.shared_proj = nn.Sequential(
            nn.Linear(2048 + num_geo, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
        )
        self.side_head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 2),
        )
        self.regression_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 4),
        )
        self.direction_head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 2),
        )
    
    def forward(self, x, geo=None):
        feats = self.backbone(x).flatten(1)
        if geo is not None: feats = torch.cat([feats, geo], dim=1)
        shared = self.shared_proj(feats)
        return self.side_head(shared), self.regression_head(shared), self.direction_head(shared)

print('Model ready')

Model ready


In [21]:
criterion_ce = nn.CrossEntropyLoss()
criterion_l1 = nn.SmoothL1Loss()
W_SIDE, W_REG, W_DIR = 1.0, 5.0, 1.0

def train_one_model(seed, all_samples, target_stats, num_epochs=NUM_EPOCHS):
    """Train one model on ALL data with given seed. Returns model path."""
    print(f'\n{"="*50}')
    print(f'Training seed={seed} on ALL {len(all_samples)} samples, {num_epochs} epochs')
    print(f'{"="*50}')
    
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    
    ds = ShadowDatasetV5(all_samples, target_stats, augment=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, 
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                        persistent_workers=True)
    
    model = ShadowModelV5().to(device)
    backbone_p = [p for n,p in model.named_parameters() if 'backbone' in n]
    head_p = [p for n,p in model.named_parameters() if 'backbone' not in n]
    optimizer = optim.AdamW([
        {'params': backbone_p, 'lr': BACKBONE_LR},
        {'params': head_p, 'lr': LR},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    
    t0 = time.time()
    for epoch in range(num_epochs):
        model.train()
        tl_sum, tn = 0, 0
        for imgs, sides, regs, dirs, geos in loader:
            imgs=imgs.to(device); sides=sides.to(device); regs=regs.to(device)
            dirs=dirs.to(device); geos=geos.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                sl,rp,dl = model(imgs, geos)
                loss = W_SIDE*criterion_ce(sl,sides) + W_REG*criterion_l1(rp,regs) + W_DIR*criterion_ce(dl,dirs)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tl_sum += loss.item()*imgs.size(0); tn += imgs.size(0)
        scheduler.step()
        
        if epoch%10==0 or epoch==num_epochs-1:
            print(f'  Ep {epoch:3d}/{num_epochs} | loss={tl_sum/tn:.4f} | {time.time()-t0:.0f}s')
    
    path = str(SAVE_DIR / f'model_seed{seed}.pth')
    torch.save(model.state_dict(), path)
    print(f'  Saved: {path} ({time.time()-t0:.0f}s total)')
    return path

print('Training function ready')

Training function ready


## Train 3 models with different seeds

In [22]:
model_paths = []
t_total = time.time()

for seed in SEEDS:
    path = train_one_model(seed, all_samples, target_stats, NUM_EPOCHS)
    model_paths.append(path)

print(f'\nAll 3 models trained in {(time.time()-t_total)/60:.1f} min')


Training seed=42 on ALL 1693 samples, 40 epochs
  Ep   0/40 | loss=2.6507 | 15s
  Ep  10/40 | loss=1.3201 | 164s
  Ep  20/40 | loss=0.9750 | 309s
  Ep  30/40 | loss=0.8000 | 447s
  Ep  39/40 | loss=0.7461 | 571s
  Saved: /home/y2a/Hackaton/shadow_detection/model_seed42.pth (571s total)

Training seed=123 on ALL 1693 samples, 40 epochs
  Ep   0/40 | loss=2.6112 | 15s
  Ep  10/40 | loss=1.3376 | 158s
  Ep  20/40 | loss=1.0008 | 302s
  Ep  30/40 | loss=0.8425 | 445s
  Ep  39/40 | loss=0.7881 | 575s
  Saved: /home/y2a/Hackaton/shadow_detection/model_seed123.pth (575s total)

Training seed=777 on ALL 1693 samples, 40 epochs
  Ep   0/40 | loss=2.8069 | 15s
  Ep  10/40 | loss=1.3085 | 157s
  Ep  20/40 | loss=1.0162 | 299s
  Ep  30/40 | loss=0.7281 | 439s
  Ep  39/40 | loss=0.5670 | 565s
  Saved: /home/y2a/Hackaton/shadow_detection/model_seed777.pth (565s total)

All 3 models trained in 28.6 min


## Ensemble prediction with TTA

In [23]:
tf = transforms.Compose([
    transforms.Resize(INPUT_SIZE), transforms.ToTensor(),
    transforms.Normalize([0.422,0.413,0.394],[0.167,0.174,0.233])])

sample_sub = pd.read_csv(SAMPLE_CSV)

def predict_one_model(model_path, test_dir, sample_sub):
    """Predict with one model + TTA. Returns raw predictions."""
    model = ShadowModelV5().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    preds = []
    with torch.no_grad():
        for _, row in sample_sub.iterrows():
            stem = row['id']
            img_path = test_dir / f'{stem}.png'
            img_raw = np.array(Image.open(img_path).convert('RGB'))
            geo = extract_geometric_features(img_raw)
            geo_t = torch.from_numpy(geo).float().unsqueeze(0).to(device)
            
            img = Image.open(img_path).convert('RGB')
            img_t = tf(img).unsqueeze(0).to(device)
            
            # Original
            with torch.amp.autocast('cuda'):
                sl1, rp1, dl1 = model(img_t, geo_t)
            
            # TTA flip
            img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
            img_flip_t = tf(img_flip).unsqueeze(0).to(device)
            geo_flip = geo.copy()
            geo_flip[0],geo_flip[1]=geo[1],geo[0]; geo_flip[2]=geo_flip[0]/(geo_flip[1]+1e-6)
            geo_flip[3]=1-geo[3]; geo_flip[8]=1-geo[8]; geo_flip[9]=1-geo[9]; geo_flip[10]=1-geo[10]
            geo_flip[11],geo_flip[12]=geo[12],geo[11]
            geo_flip_t = torch.from_numpy(geo_flip).float().unsqueeze(0).to(device)
            
            with torch.amp.autocast('cuda'):
                sl2, rp2, dl2 = model(img_flip_t, geo_flip_t)
            
            # Average regression
            rp_avg = ((rp1 + rp2) / 2.0).float().cpu().numpy()[0]
            
            # Side: average + un-flip
            sp1 = F.softmax(sl1.float(), dim=1).cpu().numpy()[0]
            sp2 = F.softmax(sl2.float(), dim=1).cpu().numpy()[0]
            sp_avg = (sp1 + sp2[::-1]) / 2
            
            # Direction: average + un-flip  
            dp1 = F.softmax(dl1.float(), dim=1).cpu().numpy()[0]
            dp2 = F.softmax(dl2.float(), dim=1).cpu().numpy()[0]
            dp_avg = (dp1 + dp2[::-1]) / 2
            
            preds.append({
                'id': stem,
                'rp': rp_avg,       # raw regression (normalized)
                'sp': sp_avg,       # side probabilities [p_left, p_right]
                'dp': dp_avg,       # direction probabilities
            })
    return preds

# Predict with all 3 models
all_preds = []
for i, mp in enumerate(model_paths):
    print(f'Predicting with model {i+1}/3...')
    preds = predict_one_model(mp, TEST_DIR, sample_sub)
    all_preds.append(preds)
    print(f'  Done ({len(preds)} predictions)')

print('All predictions done')

Predicting with model 1/3...


/tmp/ipykernel_166190/981688733.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


  Done (414 predictions)
Predicting with model 2/3...


/tmp/ipykernel_166190/981688733.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


  Done (414 predictions)
Predicting with model 3/3...


/tmp/ipykernel_166190/981688733.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


  Done (414 predictions)
All predictions done


In [24]:
# Ensemble: average all 3 models' predictions
predictions = []
n_models = len(all_preds)

for j in range(len(all_preds[0])):
    stem = all_preds[0][j]['id']
    
    # Average regression predictions
    rp_ens = np.mean([all_preds[m][j]['rp'] for m in range(n_models)], axis=0)
    # Average side probabilities
    sp_ens = np.mean([all_preds[m][j]['sp'] for m in range(n_models)], axis=0)
    # Average direction probabilities
    dp_ens = np.mean([all_preds[m][j]['dp'] for m in range(n_models)], axis=0)
    
    side = int(sp_ens.argmax())
    
    dist = denorm_t(rp_ens[0], 'distance_from_edge')
    bw = denorm_t(rp_ens[1], 'bbox_width')
    bh = denorm_t(rp_ens[2], 'bbox_height')
    yc = denorm_t(rp_ens[3], 'y_center')
    
    if side == 0:
        xmin = -dist; xmax = xmin + bw
    else:
        xmax = IMG_W + dist; xmin = xmax - bw
    ymin = yc - bh/2; ymax = yc + bh/2
    
    # Direction: abstain if not confident enough
    dir_conf = max(dp_ens)
    if dir_conf > 0.6:
        direction = int(dp_ens.argmax())
    else:
        direction = -1
    
    predictions.append({'id': stem, 'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax, 'direction': direction})

pred_df = pd.DataFrame(predictions)
submission = sample_sub[['id']].merge(pred_df, on='id', how='left')
submission['direction'] = submission['direction'].astype(int)

out_path = str(SAVE_DIR / 'submission_v5_ensemble.csv')
submission.to_csv(out_path, index=False)

n_in = (submission['direction']==1).sum()
n_out = (submission['direction']==0).sum()
n_abs = (submission['direction']==-1).sum()
print(f'\nEnsemble submission saved: {out_path}')
print(f'Rows: {len(submission)}')
print(f'Direction: {n_in} into, {n_out} out, {n_abs} abstain')
print(f'\nTheir best: 0.590')
print(f'Our V4: 0.586')
print(f'This V5 ensemble: submit and find out!')


Ensemble submission saved: /home/y2a/Hackaton/shadow_detection/submission_v5_ensemble.csv
Rows: 414
Direction: 45 into, 93 out, 276 abstain

Their best: 0.590
Our V4: 0.586
This V5 ensemble: submit and find out!


In [25]:
# Also save individual model submissions (in case ensemble doesn't help)
for i, mp in enumerate(model_paths):
    preds_single = []
    for j in range(len(all_preds[i])):
        p = all_preds[i][j]
        side = int(p['sp'].argmax())
        dist = denorm_t(p['rp'][0], 'distance_from_edge')
        bw = denorm_t(p['rp'][1], 'bbox_width')
        bh = denorm_t(p['rp'][2], 'bbox_height')
        yc = denorm_t(p['rp'][3], 'y_center')
        if side == 0: xmin=-dist; xmax=xmin+bw
        else: xmax=IMG_W+dist; xmin=xmax-bw
        ymin=yc-bh/2; ymax=yc+bh/2
        dc = max(p['dp'])
        direction = int(p['dp'].argmax()) if dc > 0.6 else -1
        preds_single.append({'id':p['id'],'xmin':xmin,'ymin':ymin,'xmax':xmax,'ymax':ymax,'direction':direction})
    
    sub_s = sample_sub[['id']].merge(pd.DataFrame(preds_single), on='id', how='left')
    sub_s['direction'] = sub_s['direction'].astype(int)
    sub_s.to_csv(str(SAVE_DIR / f'submission_v5_seed{SEEDS[i]}.csv'), index=False)
    print(f'Saved individual: submission_v5_seed{SEEDS[i]}.csv')

print('\nAll submissions saved. Pick the best one!')

Saved individual: submission_v5_seed42.csv
Saved individual: submission_v5_seed123.csv
Saved individual: submission_v5_seed777.csv

All submissions saved. Pick the best one!


In [26]:
import pandas as pd

s1 = pd.read_csv('Hackaton/shadow_detection/submission_v5_ensemble.csv')  # ours, 0.626
s2 = pd.read_csv('submission (3).csv')          # the other sub-team's best, 0.601

final = s1[['id']].copy()
for col in ['xmin','ymin','xmax','ymax']:
    final[col] = (s1[col] * 0.7 + s2[col] * 0.3)  # ours weighs more because it scored better
final['direction'] = -1

final.to_csv('submission_mega.csv', index=False)
print(f'Saved {len(final)} rows')

FileNotFoundError: [Errno 2] No such file or directory: 'submission (3).csv'

In [27]:
import pandas as pd

# Every submission we produced
files = {
    'v5_ensemble': 'submission_v5_ensemble.csv',    # 0.626
    'v5_seed123': 'submission_v5_seed123.csv',       # 0.620
    'v5_seed42': 'submission_v5_seed42.csv',         # 0.618
    'v4_fullres': 'submission_v4_fullres.csv',       # 0.614
    'v5_seed777': 'submission_v5_seed777.csv',       # 0.604
}

dfs = {name: pd.read_csv(f'/home/y2a/Hackaton/shadow_detection/{path}') for name, path in files.items()}

# Weighted average -- better models weigh more
weights = {
    'v5_ensemble': 3.0,  # best
    'v5_seed123': 2.0,
    'v5_seed42': 1.5,
    'v4_fullres': 1.5,
    'v5_seed777': 1.0,
}

base = list(dfs.values())[0]
final = base[['id']].copy()

for col in ['xmin','ymin','xmax','ymax']:
    total_w = sum(weights.values())
    final[col] = sum(dfs[name][col] * weights[name] for name in dfs) / total_w

final['direction'] = -1

final.to_csv('/home/y2a/Hackaton/shadow_detection/submission_MEGA.csv', index=False)
print(f'Saved! {len(final)} rows')
print('SUBMIT THIS!')

Saved! 414 rows
SUBMIT THIS!
